# Flood probabilities, frequencies and thresholds

**Time:** 40 minutes · **Prerequisites:** basic Python; run cells from top to bottom.

Separate event rates from probabilities; calculate cumulative exposure; test how a threshold changes a headline count.

[Run this notebook in JupyterLite](https://jltobias.github.io/JupyterLite-Sea-Level-Rise/lite/lab/index.html?path=notebooks/03_probability_and_thresholds.ipynb) · [Full-screen dashboard](https://jltobias.github.io/JupyterLite-Sea-Level-Rise/dashboard/) · [Official poster](https://plus.iasociety.org/sites/default/files/2024-09/e-poster_755.pdf)

Each notebook runs independently. Use **Run → Run All Cells**; the first package download can take a minute. Save a downloaded copy to keep your work. The read-only book includes executed examples.

In [1]:
# First run downloads Python packages in JupyterLite; subsequent runs use browser caches.
import sys
if sys.platform == 'emscripten':
    import piplite
    await piplite.install(['numpy', 'pandas', 'matplotlib', 'plotly', 'ipywidgets'])
from pathlib import Path
root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(root))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import plotly.express as px
from IPython.display import display, HTML
from coastlab import (DATA, SCENARIOS, YEARS, COLORS, load_data, select, cards,
                      toy_slr, toy_probability, cumulative_probability,
                      haversine_km, show, dashboard, land_traces)
facilities, exposure = load_data()
published = pd.read_csv(DATA / 'poster_aggregates.csv')
print('Ready: public poster aggregates + clearly labeled fictional facility fixtures.')

Ready: public poster aggregates + clearly labeled fictional facility fixtures.


## Start with the unit

Annual exceedance probability $p$ is bounded by 0 and 1. An expected annual number of events $\lambda$ can exceed 1. The source workbook's EAE fields contain values above 1, so treating them directly as probability fractions is incorrect. Their full definitions, time windows and category rules need the original PAT metadata.

Under an explicitly assumed Poisson process, $P(N\geq1)=1-e^{-\lambda}$. That conversion is a model assumption, not a universal interpretation of an undocumented column.

For independent annual exceedance indicators with probabilities $p_y$, the probability of at least one event is $1-\prod_y(1-p_y)$. Independence may be unrealistic, and a changing climate makes constant annual probabilities a weak long-term assumption. A return period $T=1/p$ is an average recurrence concept under stationarity, not a timetable.

In [2]:
p = 0.10
display(pd.DataFrame({'years':[1,10,30], 'at_least_one':[cumulative_probability([p]*n) for n in [1,10,30]]}))
rates=np.array([.01,.1,1,12])
display(pd.DataFrame({'assumed_Poisson_events_per_year':rates,'probability_at_least_one':1-np.exp(-rates)}))
assert np.isclose(cumulative_probability([.1]*10),1-.9**10)

,years,at_least_one
0,1,0.100000
1,10,0.651322
2,30,0.957609


,assumed_Poisson_events_per_year,probability_at_least_one
0,0.01,0.009950
1,0.10,0.095163
2,1.00,0.632121
3,12.00,0.999994


## A classification is a decision rule

The fictional lab uses a logistic response: $p=1/(1+\exp((e-s-u)/a))$, with invented elevation $e$, sea-level increment $s$, surge offset $u=0.7$ m and scale $a=0.55$ m. This helps reveal how small height changes can affect a threshold. It has no calibration to the real portfolio.

The slider classifies a site as exposed when $p\geq\tau$. Changing $\tau$ changes the count even though the underlying probabilities have not changed. The denominator remains all facilities in the selected cohort.

In [3]:
rows=select(facilities,exposure,scenario='RCP 8.5',year=2100)
thresholds=np.linspace(.01,.99,99)
fig=go.Figure(go.Scatter(x=thresholds,y=[cards(rows,t)['above_threshold'] for t in thresholds],mode='lines'))
fig.update_layout(title='Fictional count is sensitive to the decision threshold',xaxis_title='Annual probability threshold',yaxis_title='Facilities above threshold')
show(fig)
print('Sum of marginal probabilities (expected exposed facilities):',round(rows.annual_probability.sum(),2))
print('Threshold count:',cards(rows,.1)['above_threshold'])

<iframe title="Interactive teaching figure" style="width:100%;height:560px;border:0" srcdoc="<!doctype html>
<html>
<head>
 <meta charset="utf-8" />
 <style>html, body {height: 100%;}</style>
</head>
<body>
 <div style="height:100%; width:100%;"> <script>window.PlotlyConfig = {MathJaxConfig: 'local'};</script>
 <script charset="utf-8" src="../../assets/plotly.min.js"></script> <div id="e08d4115-c0e3-4420-b918-ca377fde9867" class="plotly-graph-div" style="height:100%; width:100%;"></div> <script> window.PLOTLYENV=window.PLOTLYENV || {}; if (document.getElementById("e08d4115-c0e3-4420-b918-ca377fde9867")) { Plotly.newPlot( "e08d4115-c0e3-4420-b918-ca377fde9867", [{"mode":"lines","x":{"dtype":"f8","bdata":"exSuR+F6hD97FK5H4XqUP7gehetRuJ4\u002fexSuR+F6pD+amZmZmZmpP7kehetRuK4\u002f61G4HoXrsT97FK5H4Xq0PwrXo3A9Crc\u002fmZmZmZmZuT8pXI\u002fC9Si8P7gehetRuL4\u002fpHA9CtejwD\u002fsUbgehevBPzQzMzMzM8M\u002fexSuR+F6xD\u002fD9Shcj8LFPwvXo3A9Csc\u002fUrgehetRyD+amZmZmZnJP+J6FK5H4co\u002fKVyPwvUozD9xPQrXo3DNP7kehetRuM4\u002fAAAAAAAA0D+kcD0K16PQP0jhehSuR9E\u002f7FG4HoXr0T+QwvUoXI\u002fSPzMzMzMzM9M\u002f16NwPQrX0z97FK5H4XrUPx+F61G4HtU\u002fw\u002fUoXI\u002fC1T9nZmZmZmbWPwvXo3A9Ctc\u002frkfhehSu1z9SuB6F61HYP\u002fYoXI\u002fC9dg\u002fmpmZmZmZ2T8+CtejcD3aP+J6FK5H4do\u002fhetRuB6F2z8pXI\u002fC9SjcP83MzMzMzNw\u002fcT0K16Nw3T8VrkfhehTeP7kehetRuN4\u002fXI\u002fC9Shc3z8AAAAAAADgP1K4HoXrUeA\u002fpHA9Ctej4D\u002f2KFyPwvXgP0jhehSuR+E\u002fmpmZmZmZ4T\u002fsUbgehevhPz4K16NwPeI\u002fkML1KFyP4j\u002fhehSuR+HiPzMzMzMzM+M\u002fhetRuB6F4z\u002fXo3A9CtfjPylcj8L1KOQ\u002fexSuR+F65D\u002fNzMzMzMzkPx+F61G4HuU\u002fcT0K16Nw5T\u002fD9Shcj8LlPxWuR+F6FOY\u002fZ2ZmZmZm5j+5HoXrUbjmPwrXo3A9Cuc\u002fXI\u002fC9Shc5z+uR+F6FK7nPwAAAAAAAOg\u002fUrgehetR6D+kcD0K16PoP\u002fYoXI\u002fC9eg\u002fSOF6FK5H6T+amZmZmZnpP+xRuB6F6+k\u002fPgrXo3A96j+QwvUoXI\u002fqP+J6FK5H4eo\u002fMzMzMzMz6z+F61G4HoXrP9ejcD0K1+s\u002fKVyPwvUo7D97FK5H4XrsP83MzMzMzOw\u002fH4XrUbge7T9xPQrXo3DtP8P1KFyPwu0\u002fFa5H4XoU7j9nZmZmZmbuP7kehetRuO4\u002fCtejcD0K7z9cj8L1KFzvP65H4XoUru8\u002f"},"y":[173,156,144,138,131,123,120,117,115,112,111,110,109,106,105,105,102,101,99,94,93,88,86,82,81,81,81,80,77,75,75,74,73,73,71,69,68,68,68,66,66,64,64,64,63,62,61,60,59,57,55,53,53,51,50,48,47,46,45,44,43,42,42,40,38,38,35,33,33,32,31,30,27,27,26,26,25,22,21,21,20,19,17,14,12,12,11,9,7,5,2,0,0,0,0,0,0,0,0],"type":"scatter"}], {"template":{"data":{"histogram2dcontour":[{"type":"histogram2dcontour","colorbar":{"outlinewidth":0,"ticks":""},"colorscale":[[0.0,"#0d0887"],[0.1111111111111111,"#46039f"],[0.2222222222222222,"#7201a8"],[0.3333333333333333,"#9c179e"],[0.4444444444444444,"#bd3786"],[0.5555555555555556,"#d8576b"],[0.6666666666666666,"#ed7953"],[0.7777777777777778,"#fb9f3a"],[0.8888888888888888,"#fdca26"],[1.0,"#f0f921"]]}],"choropleth":[{"type":"choropleth","colorbar":{"outlinewidth":0,"ticks":""}}],"histogram2d":[{"type":"histogram2d","colorbar":{"outlinewidth":0,"ticks":""},"colorscale":[[0.0,"#0d0887"],[0.1111111111111111,"#46039f"],[0.2222222222222222,"#7201a8"],[0.3333333333333333,"#9c179e"],[0.4444444444444444,"#bd3786"],[0.5555555555555556,"#d8576b"],[0.6666666666666666,"#ed7953"],[0.7777777777777778,"#fb9f3a"],[0.8888888888888888,"#fdca26"],[1.0,"#f0f921"]]}],"heatmap":[{"type":"heatmap","colorbar":{"outlinewidth":0,"ticks":""},"colorscale":[[0.0,"#0d0887"],[0.1111111111111111,"#46039f"],[0.2222222222222222,"#7201a8"],[0.3333333333333333,"#9c179e"],[0.4444444444444444,"#bd3786"],[0.5555555555555556,"#d8576b"],[0.6666666666666666,"#ed7953"],[0.7777777777777778,"#fb9f3a"],[0.8888888888888888,"#fdca26"],[1.0,"#f0f921"]]}],"contourcarpet":[{"type":"contourcarpet","colorbar":{"outlinewidth":0,"ticks":""}}],"contour":[{"type":"contour","colorbar":{"outlinewidth":0,"ticks":""},"colorscale":[[0.0,"#0d0887"],[0.1111111111111111,"#46039f"],[0.2222222222222222,"#7201a8"],[0.3333333333333333,"#9c179e"],[0.4444444444444444,"#bd3786"],[0.5555555555555556,"#d8576b"],[0.6666666666666666,"#ed7953"],[0.7777777777777778,"#fb9f3a"],[0.

Sum of marginal probabilities (expected exposed facilities): 58.7
Threshold count: 112


In [4]:
# Annual interpolation is explicit; nine decadal values are NOT nine decades of annual data.
example = exposure.query("facility_id == 'DEMO-01-001' and scenario == 'RCP 8.5'").sort_values('year')
annual_years=np.arange(2030,2061)
annual_p=np.interp(annual_years,example.year,example.annual_probability)
print('Illustrative 2030–2060 cumulative probability, independent years:',cumulative_probability(annual_p))

Illustrative 2030–2060 cumulative probability, independent years: 0.9999907484957657


## Lab: interrogate uncertainty

Change the surge offset by ±0.25 m and compare counts at thresholds 1%, 10% and 50%. Then describe which is uncertainty about nature and which is a decision-maker's threshold.

**Worked check:** At fixed probabilities, increasing the threshold cannot increase the number classified as exposed. The sum of marginal probabilities is an expected number of exposed sites; it does not require independence between sites. Cumulative probability across years does require a dependence assumption for the product formula.

**Deliverable:** A sensitivity table with units and a paragraph stating the independence assumption. Do not add probabilities across years or interpret modeled exposed sites as affected patients.